In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

print(os.getcwd())
tfrecordpath = "../Data/tfrecords/"


2025-02-05 18:42:12.071191: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738777332.083539   61577 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738777332.086955   61577 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-05 18:42:12.100183: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


/mnt/c/Users/alexs/Desktop/levbot/Training


### Load in data
#### Define schema


In [2]:
def decode(record_bytes):
    # Function for parsing each record in the tf files
    example = tf.io.parse_single_example(
        # Data
        record_bytes,

        # Schema
        {
        'Timeframe': tf.io.FixedLenFeature([], tf.string),
        'timestamp': tf.io.RaggedFeature(dtype=tf.int64),
        'Open': tf.io.RaggedFeature(dtype=tf.float32),
        'High': tf.io.RaggedFeature(dtype=tf.float32),
        'Low': tf.io.RaggedFeature(dtype=tf.float32),
        'Close': tf.io.RaggedFeature(dtype=tf.float32),
        'Volume': tf.io.RaggedFeature(dtype=tf.float32),
        }
        )

    return example

In [3]:
def getDataset(path):
    ds = tf.data.TFRecordDataset(path,  num_parallel_reads = tf.data.AUTOTUNE)
    ds = ds.map(decode, num_parallel_calls = tf.data.AUTOTUNE)
    return ds



In [4]:
import TensorSlider

In [5]:
windowsize = 100
lookahead = 5

def slider():


    path = tfrecordpath + "BTCUSD_PERP/{tframe}.tfrecord"
    datasets = {"1m" : getDataset(path.format(tframe="1m")),
                "5m":  getDataset(path.format(tframe="5m")),
                "15m":  getDataset(path.format(tframe="15m")),
                "30m":  getDataset(path.format(tframe="30m")),
                "1h":  getDataset(path.format(tframe="1h"))}
    return TensorSlider.WindowSlider(datasets, windowsize, lookahead, batchsize=25)

dataset = tf.data.Dataset.from_generator(slider, 
    output_signature=(
        tf.TensorSpec((5,5,windowsize), dtype=tf.float32),
        tf.TensorSpec((lookahead+1,5), dtype=tf.float32))
    )

I0000 00:00:1738777333.874732   61577 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5592 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:05:00.0, compute capability: 8.6


In [16]:
ws = slider()
print(ws.__iter__().__next__()[0].shape)

(25, 5, 5, 100)


In [21]:
def createLabelsbatch(data, lookforward):
    """
    create input labels from the lookahead data
    """
    print(data.shape)
    print(lookforward.shape)
    high = []
    low = []

    for batch in lookforward:
        ohlc = batch
        ohlc /= ohlc[0,:] # divide by open price
        ohlc -= 1 # zero out
        ohlc *= 100 # convert to 1/10 percentage, so 1 = 10 percent
        ohlc = tf.clip_by_value(ohlc,-1,1)

        high.append(ohlc[tf.math.argmax(ohlc[:,1]), 1])
        low.append(ohlc[tf.math.argmin(ohlc[:,2]), 2])

    datares = []

    # same process, compute relative deltas
    for batch in data:
        processed = []
        latestClose = batch[0,3,0] # base timeframe
        pricemult = 10 # 1 = 10 percent
        volmult = 0.1
        for i in range(0, batch.shape[0]):
            # lets go timeframe by timeframe
            prices = tf.divide(batch[i,0:4,:], latestClose) - 1
            prices = tf.multiply(prices, pricemult)
            volumes = tf.divide(batch[i,4,:], batch[i,4,0]) - 1
            volumes = tf.multiply(volumes, volmult)
            volumes = tf.expand_dims(volumes, 0)
            processed.append(tf.concat([prices, volumes], axis=0))

        datares.append(tf.clip_by_value(tf.stack(processed, axis=0), clip_value_min=-1, clip_value_max=1))

    # remove nans
    low = tf.stack(*low)
    high = tf.stack(*high)
    datares = tf.stack(*datares)
    datares = tf.keras.ops.nan_to_num(datares)

    return datares, tf.concat([low, high], axis=0)




In [18]:
def createLabels(data, lookforward):
    """
    create input labels from the lookaehad data
    """

    ohlc = lookforward
    ohlc /= ohlc[0,:] # divide by open price
    ohlc -= 1 # zero out
    ohlc *= 100 # convert to 1/10 percentage, so 1 = 10 percent
    ohlc = tf.clip_by_value(ohlc,-1,1)
    high = ohlc[tf.math.argmax(ohlc[:,1]), 1]
    low = ohlc[tf.math.argmin(ohlc[:,2]), 2]


    # same process, compute relative deltas
    processed = []
    latestClose = data[0,3,0] # base timeframe
    pricemult = 10 # 1 = 10 percent
    volmult = 0.1
    for i in range(0, data.shape[0]):
        # lets go timeframe by timeframe
        prices = tf.divide(data[i,0:4,:], latestClose) - 1
        prices = tf.multiply(prices, pricemult)
        volumes = tf.divide(data[i,4,:], data[i,4,0]) - 1
        volumes = tf.multiply(volumes, volmult)
        volumes = tf.expand_dims(volumes, 0)
        processed.append(tf.concat([prices, volumes], axis=0))

    datares = tf.clip_by_value(tf.stack(processed, axis=0), clip_value_min=-1, clip_value_max=1)

    # remove nans
    datares = tf.keras.ops.nan_to_num(datares)

    return tf.expand_dims(datares,0), [low, high]

In [19]:
def printtimeframe(features):
    feet = features[0]
    print(feet.shape)
    for ft in feet:
        plt.plot(ft[4])
    plt.show()

In [20]:
import time
t0 = None
for i, data in enumerate(dataset.map(createLabels)):
    print(data.shape)
    if np.any((tf.math.is_nan(data[0]))):
        print("NaN in data")
    if i == 0:
        t0 = time.time()
    if i > 200:
        tdelta = time.time() - t0
        print(f"Time taken for {i+1} calls: {tdelta}s, thats {((tdelta/(i+1))*1000):.2f}ms per call")
        print(data[1])
        printtimeframe(data[0])
        break
        

2025-02-05 18:47:02.299746: W tensorflow/core/framework/op_kernel.cc:1829] INVALID_ARGUMENT: TypeError: `generator` yielded an element of shape (25, 5, 5, 100) where an element of shape (5, 5, 100) was expected.
Traceback (most recent call last):

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/ops/script_ops.py", line 269, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/data/ops/from_generator_op.py", line 235, in generator_py_func
    raise TypeError(

TypeError: `generator` yielded an element of shape (25, 5, 5, 100) where an element of shape (5, 5, 100) was expected.


2025-02-05 18:47:02.299897: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvo

InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} TypeError: `generator` yielded an element of shape (25, 5, 5, 100) where an element of shape (5, 5, 100) was expected.
Traceback (most recent call last):

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/ops/script_ops.py", line 269, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/data/ops/from_generator_op.py", line 235, in generator_py_func
    raise TypeError(

TypeError: `generator` yielded an element of shape (25, 5, 5, 100) where an element of shape (5, 5, 100) was expected.


	 [[{{node PyFunc}}]] [Op:IteratorGetNext] name: 

In [10]:
import keras
path = "models/modeltest/model.keras"
model = keras.models.load_model(path)
history = model.fit(dataset.map(createLabels),
                        epochs=1, verbose=1,
                        validation_data=None, callbacks=None)

   3732/Unknown 93s 22ms/step - MeanAbsolutePercentageError: 23716278.0000 - MeanSquaredError: 0.0276 - loss: 0.0276

KeyboardInterrupt: 